# core

> The error type, the environment convention, and the one place that decides which model runs what.

Every other module depends on this one, and it depends on nothing in the package. Three small things live here because everything needs them. `AgentError`, `agent_err` and `env`. Then the model policy, which is the interesting part: not just *which model*, but which model *for what job*.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_ne, test_fail

## Errors and environment

A harness fails in two ways, and a person needs to be told them differently. `AgentError` is a refusal. Something the harness will not do, which the user can act on. Anything else is a bug, and should look like one.

In [ ]:
#| export
import functools, importlib, importlib.util, json, os, platform, re, shutil, subprocess, sys, time
from fastcore.all import Path
from dataclasses import dataclass, field

In [ ]:
#| export
ENV_PREFIX, ENV_FALLBACK = 'RAMABANA_', 'LEELA_'

class AgentError(Exception):
    "Something the harness refuses to do, rather than a failure while doing it."

def agent_err(e):
    "A caught exception for a user-facing harness surface."
    return f'{type(e).__name__}: {e}'

def use_env_prefix(prefix, fallback=None):
    """Name the environment variables this application reads, most specific first.
    `use_env_prefix('LEELA_', 'RAMABANA_')`
    """
    global ENV_PREFIX, ENV_FALLBACK
    ENV_PREFIX = prefix if prefix.endswith('_') else prefix + '_'
    if fallback is not None: ENV_FALLBACK = fallback if fallback.endswith('_') else fallback + '_'
    return ENV_PREFIX, ENV_FALLBACK


def env(name, dflt=None):
    "`$<prefix><name>`, then `$<fallback><name>`, then `dflt`. See `use_env_prefix`."
    return os.environ.get(ENV_PREFIX+name) or os.environ.get(ENV_FALLBACK+name) or dflt

`agent_err` renders a caught exception for a surface a person reads, keeping the type name because "no such file" without `FileNotFoundError` in front of it explains nothing.

In [ ]:
agent_err(FileNotFoundError('nbs/99_missing.ipynb'))

'FileNotFoundError: nbs/99_missing.ipynb'

`env` reads `$RAMABANA_<name>` first and `$LEELA_<name>` second. A Leela user's existing configuration keeps working while the new prefix takes precedence.

In [ ]:
os.environ['LEELA_MODEL'] = 'opus'
env('MODEL')

'opus'

In [ ]:
os.environ['RAMABANA_MODEL'] = 'sonnet'
test_eq(env('MODEL'), 'sonnet')
del os.environ['RAMABANA_MODEL'], os.environ['LEELA_MODEL']
env('MODEL', 'gemma-e4b')

'gemma-e4b'

## The model tables

`MODELS` maps shared short names to LiteRT models on device or cloud model ids. `JOBS` is the set of distinct jobs the policy can route separately.

In [ ]:
#| export
JOBS = ('turn', 'oneshot', 'inline', 'completion', 'classify', 'summary', 'subagent')

#: One-shot jobs share the `oneshot` policy unless set individually. Not `turn` or `subagent`.
ONESHOT_JOBS = ('oneshot', 'completion', 'classify', 'summary', 'inline')
LOCAL = {'gemma-e2b': 'litert-community/gemma-4-E2B-it-litert-lm',
    'gemma-e4b': 'litert-community/gemma-4-E4B-it-litert-lm',
    'gemma-12b': 'litert-community/gemma-4-12B-it-litert-lm'}
MLX = {'qwen-4b': 'mlx-community/Qwen3.5-4B-MLX-4bit',
       'mini-coder-4b': 'mlx-community/mini-coder-4b-OptiQ-4bit',
       'ornith-9b': 'mlx-community/Ornith-1.0-9B-8bit'}
LLAMA = {'llama-qwen-0.6b': 'Qwen/Qwen3-0.6B-GGUF',
         'llama-qwen-1.7b': 'Qwen/Qwen3-1.7B-GGUF',
         'llama-qwen-4b': 'Qwen/Qwen3-4B-GGUF'}

GPT = {name: f'openai/{name}' for name in (
    'gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano',
    'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.6', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra')}
GPT.update({name: f'codex/{name}' for name in (
    'gpt-5.3-codex-spark', 'gpt-5.5')})
CLAUDE = {name: f'claude_code/{name}' for name in (
    'claude-haiku-4-5', 'claude-sonnet-4-5', 'claude-sonnet-4-6',
    'claude-opus-4-6', 'claude-opus-4-8', 'claude-fable-5',
    'claude-sonnet-5', 'claude-opus-5')}
CLOUD = {**GPT, **CLAUDE,
    'sonnet': CLAUDE['claude-sonnet-5'], 'opus': CLAUDE['claude-opus-5'],
    'fable': CLAUDE['claude-fable-5'], 'gpt': GPT['gpt-5.6-terra'],
    'gpt-mini': GPT['gpt-5.6-luna'], 'gpt-sol': GPT['gpt-5.6-sol']}

#: Cursor's own catalogue, under the ids Cursor accepts.
CURSOR = {f'cursor/{mid}': mid for mid in (
    'grok-4.5', 'composer-2.5', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
    'gpt-5.6-sol', 'gpt-5.6-terra', 'gemini-3.6-flash', 'kimi-k3', 'glm-5.2')}

#: Claude Code's catalogue, prefixed the same way and for the same reason
CLAUDE = {f'claude/{mid}': mid for mid in (
    'claude-opus-5', 'claude-sonnet-5', 'claude-haiku-4-5', 'claude-fable-5',
    'claude-opus-4-8', 'claude-sonnet-4-6')}

DFLT_AGENT_CTX = 128_000
RUNTIMES = ('litert', 'mlx', 'llama', 'cursor', 'claude', 'copilot', 'remote')
AGENTS = ('cursor', 'claude')
HOSTED = ('remote', 'copilot', *AGENTS)
COPILOT_UNAVAILABLE = ('copilot runtime is unavailable; install rishi[copilot], then sign in to '
    'Copilot in an editor or run `python -c "from rishi.copilot import copilot_login; copilot_login()"`')
CUSTOM = {}
_RUNTIME_DEPS = {'litert': 'litert_lm', 'mlx': 'mlx_lm', 'llama': 'llama_cpp'}

def _harness_available(mod, binary):
    "Whether an agent harness can be reached at all: its SDK, or the binary it drives."
    try: m = importlib.import_module(mod)
    except Exception: return False
    try:
        if m.sdk_available(): return True
    except Exception: pass
    try: return bool(getattr(m, binary)())
    except Exception: return False

def cursor_mode(config=None, tools=True):
    "The Cursor mode a spec runs in: `agent` where there are tools, because a read-only mode will not run one."
    return (config or {}).get('mode') or ('agent' if tools else 'ask')

def _agent_native(runtime, spec=None):
    """Whether this harness will carry the tool schemas itself, for a spec shaped like this one.
    Cursor will, through its SDK's custom tools, and only in `agent` mode """
    if runtime != 'cursor': return False
    try:
        m = importlib.import_module('rishi.cursor')
        cfg = getattr(spec, 'config', None) or {}
        return (bool(getattr(m.CursorChat, 'tool_channel', None)) and bool(m.sdk_available(cfg.get('api_key')))
                and m.sdk_mode(cursor_mode(cfg)) == 'agent')
    except Exception: return False

def _cursor_available(): return _harness_available('rishi.cursor', 'cursor_bin')
def _claude_available(): return _harness_available('rishi.claude', 'claude_bin')

def _copilot_available():
    """Whether Copilot can be reached: rishi imports it, and a GitHub OAuth token is on hand to
    exchange. Reads the environment and the editor config files, and never the network."""
    try:
        m = importlib.import_module('rishi.copilot')
        return bool(m.copilot_oauth())
    except Exception: return False

def runtime_available(runtime):
    "Whether Rishi's optional dependency for `runtime` can be reached. Never raises."
    if runtime == 'remote': return True
    if runtime == 'cursor': return _cursor_available()
    if runtime == 'claude': return _claude_available()
    if runtime == 'copilot': return _copilot_available()
    try: return importlib.util.find_spec(_RUNTIME_DEPS[runtime]) is not None
    except (ImportError, KeyError, ValueError): return False

MODELS = {**{k: ('litert', v) for k, v in LOCAL.items()},
          **{k: ('mlx', v) for k, v in MLX.items()},
          **{k: ('llama', v) for k, v in LLAMA.items()},
          **{k: ('cursor', v) for k, v in CURSOR.items()},
          **{k: ('claude', v) for k, v in CLAUDE.items()},
          **{k: ('remote', v) for k, v in CLOUD.items()}}

A name resolves to either the LiteRT runtime or a cloud transport.

In [ ]:
{name: MODELS[name] for name in ('gemma-e4b', 'gpt-4.1-mini', 'claude-sonnet-4-5', 'sonnet')}

{'gemma-e4b': ('litert', 'litert-community/gemma-4-E4B-it-litert-lm'),
 'gpt-4.1-mini': ('remote', 'openai/gpt-4.1-mini'),
 'claude-sonnet-4-5': ('remote', 'claude_code/claude-sonnet-4-5'),
 'sonnet': ('remote', 'claude_code/claude-sonnet-5')}

## Credentials

`auth_status` reports which credential sources fastllm could use. It reads status metadata only: no function here returns, logs or stores a secret, and the Claude Code login is probed by asking the `claude` binary, never by reading its credential file.

In [ ]:
#| export
def _json_has(path, *keys):
    "Whether nested keys in a JSON file are present and truthy."
    try:
        from fastcore.basics import nested_idx
        return bool(nested_idx(Path(path).expanduser().read_json(), *keys))
    except Exception: return False

def _claude_login():
    "Only status metadata. Credentials never leave Claude Code or enter Leela."
    if not shutil.which('claude'): return False
    try:
        p = subprocess.run(['claude', 'auth', 'status', '--json'], capture_output=True, text=True, timeout=3)
        return p.returncode == 0 and bool(json.loads(p.stdout).get('loggedIn'))
    except Exception: return False


def _install_toolslm_funccall():
    "Expose fastcore's replacement under the module name python-fastllm 0.0.36 imports."
    try: return importlib.import_module('toolslm.funccall')
    except ModuleNotFoundError as e:
        if e.name != 'toolslm.funccall': raise
    import toolslm
    funccall = importlib.import_module('fastcore.funccall')
    sys.modules['toolslm.funccall'] = funccall
    toolslm.funccall = funccall
    return funccall

def _managed_claude_mcp():
    "Whether this machine has an organisation-controlled Claude Code MCP configuration."
    paths = [Path('/Library/Application Support/ClaudeCode/managed-mcp.json'),
             Path('/etc/claude-code/managed-mcp.json')]
    if appdata := os.getenv('PROGRAMDATA'):
        paths.append(Path(appdata)/'ClaudeCode'/'managed-mcp.json')
    return any(path.exists() for path in paths)

def _claude_payload_compat(transport):
    "Make FastLLM's Claude Code payload legal wherever its MCP channel is closed."
    mk_payload = transport.mk_payload
    if getattr(mk_payload, '_ramabana_enterprise_mcp', False): return
    def compatible_payload(*args, **kwargs):
        payload = mk_payload(*args, **kwargs)
        options = payload['options']
        # the transport answers for `claude_code/` only, and carries the bare id in the options
        if tool_channel(f'claude_code/{getattr(options, "model", "")}') == 'tags':
            options.strict_mcp_config = False
            options.mcp_servers = {}
            options.allowed_tools = []
        return payload
    compatible_payload._ramabana_enterprise_mcp = True
    transport.mk_payload = compatible_payload


def claude_tags(model_id):
    "Whether this model's tools have to travel in the system prompt rather than over MCP."
    if (v := (env('CLAUDE_TAG_TOOLS') or '').strip().lower()) in ('1', 'true', 'yes'): return True
    if v in ('0', 'false', 'no'): return False
    return str(model_id or '').startswith('claude_code/') and _managed_claude_mcp()

def _load_claude_transport():
    "Load and verify FastLLM's Claude Code plugin."
    try:
        _install_toolslm_funccall()
        from fastllm.acomplete import api_registry
        _claude_payload_compat(api_registry['claude_code'])
        return True, ''
    except Exception as e: return False, agent_err(e)

def auth_status():
    'Credential sources FastLLM can use, without reading or returning any secret.'
    codex = bool(os.getenv('CODEX_AUTH_TOKEN') or _json_has(os.getenv('CODEX_AUTH_PATH', '~/.codex/auth.json'), 'tokens', 'access_token'))
    claude_transport, claude_error = _load_claude_transport()
    claude_login = _claude_login()
    copilot = _copilot_available()
    return {
        'openai': {'available': bool(os.getenv('OPENAI_API_KEY')), 'source': 'OPENAI_API_KEY'},
        'codex': {'available': codex, 'source': 'Codex login' if codex else ''},
        'anthropic': {'available': bool(os.getenv('ANTHROPIC_API_KEY')), 'source': 'ANTHROPIC_API_KEY'},
        'claude_code': {'available': claude_login and claude_transport,
                        'source': ('Claude Code login' if claude_login and claude_transport else ''),
                        'note': (f'Claude Code login found, but its FastLLM transport failed: {claude_error}'
                                 if claude_login and not claude_transport else '')},
        'gemini': {'available': bool(os.getenv('GEMINI_API_KEY')), 'source': 'GEMINI_API_KEY'},
        'copilot': {'available': copilot,
                    'source': 'GitHub Copilot sign-in' if copilot else ''},
    }

The shape is the same for every vendor whether or not it is connected. A caller can render the whole list without special cases.

In [ ]:
{k: v['available'] for k, v in auth_status().items()}

{'openai': True,
 'codex': True,
 'anthropic': False,
 'claude_code': True,
 'gemini': True}

## Listing models

`available_models` is what a model picker renders. It lists the on-device models unconditionally, adds llama.cpp only when that package is importable, and adds a vendor's cloud catalog only when a credential for it exists. The list never offers a model the process cannot actually run.

In [ ]:
#| export
_oai_cache = (0.0, [])
def _openai_models(include_legacy=False):
    "Canonical models the current OpenAI key can list. Older coding models are opt-in."
    global _oai_cache
    ids = _oai_cache[1] if (time.time() - _oai_cache[0]) < 300 else None
    if not (key := os.getenv('OPENAI_API_KEY')): return []
    if ids is None:
        try:
            import httpx2 as httpx
            r = httpx.get('https://api.openai.com/v1/models', headers={'Authorization': f'Bearer {key}'}, timeout=10)
            r.raise_for_status()
            ids = [x.get('id', '') for x in r.json().get('data', [])]
        except Exception: ids = []
        _oai_cache = (time.time(), ids)
    current = re.compile(r'^(?:gpt-5(?:\.\d+)?(?:-(?:mini|nano|pro|codex(?:-mini|-max)?|chat-latest|search-api|[a-z]+))?|o[34](?:-mini|-pro)?)$')
    legacy = re.compile(r'^gpt-4\.1(?:-mini|-nano)?$')
    return sorted({x for x in ids if (current.match(x) or (include_legacy and legacy.match(x))) and not re.search(r'-20\d\d-', x)})

_copilot_cat = (0., {})
def copilot_catalog(ttl=300):
    "Copilot's catalogue for this account, cached: `{id: entry}`, or `{}` when it cannot be reached."
    global _copilot_cat
    if (time.time() - _copilot_cat[0]) < ttl: return _copilot_cat[1]
    try:
        from rishi.copilot import copilot_catalog as cat
        d = cat()
    except Exception: d = {}
    _copilot_cat = (time.time(), d)
    return d

def _copilot_chat_models():
    "Chat ids this Copilot plan can reach. Per-plan and it moves. It is asked for, never tabled."
    return [i for i, m in copilot_catalog().items()
            if (m.get('capabilities') or {}).get('type') == 'chat']

def available_models(include_legacy=False):
    "Models selectable here. Specialized older generations appear only when requested."
    rows = []
    for runtime, models in (('litert', LOCAL), ('mlx', MLX), ('llama', LLAMA)):
        if not runtime_available(runtime): continue
        rows += [{'value': name, 'label': name, 'provider': runtime,
                  'source': f'on device via Rishi {runtime}'} for name in models]
    if runtime_available('cursor'):
        rows += [{'value': name, 'label': mid, 'provider': 'cursor',
                  'source': 'Cursor agent (CLI or SDK)'} for name, mid in CURSOR.items()]
    if runtime_available('claude'):
        rows += [{'value': name, 'label': mid, 'provider': 'claude',
                  'source': 'Claude Code (CLI or SDK)'} for name, mid in CLAUDE.items()]
    if runtime_available('copilot'):
        for model in _copilot_chat_models():
            rows.append({'value': f'copilot/{model}', 'label': model, 'provider': 'copilot',
                         'source': 'GitHub Copilot subscription'})
    auth = auth_status()
    if auth['openai']['available']:
        for model in _openai_models(include_legacy):
            rows.append({'value': f'openai/{model}', 'label': model, 'provider': 'openai',
                         'source': auth['openai']['source']})
    try:
        from fastllm.types import model_info_registry
        vendors = ('openai', 'codex', 'claude_code', 'gemini')
        for vendor in vendors:
            if not auth.get(vendor, {}).get('available'): continue
            for v, model in model_info_registry:
                if v != vendor: continue
                if not include_legacy and re.match(r'^gpt-4(?:\.|-|$)', model): continue
                rows.append({'value': f'{vendor}/{model}', 'label': model, 'provider': vendor,
                             'source': auth[vendor]['source']})
    except Exception: pass
    if auth['anthropic']['available']:
        catalog = set()
        try:
            from fastllm.types import model_info_registry
            catalog = {model for vendor, model in model_info_registry if vendor in ('anthropic', 'claude_code') and model.startswith('claude-')}
        except Exception: pass
        catalog.update(model.split('/', 1)[1] for model in CLOUD.values() if model.startswith('anthropic/'))
        for model in sorted(catalog): rows.append({'value': f'anthropic/{model}', 'label': model,
                                                   'provider': 'anthropic', 'source': auth['anthropic']['source']})
    seen, out = set(), []
    for row in rows:
        if row['value'] in seen: continue
        seen.add(row['value']); out.append(row)
    return out

Every row carries the `source` that made it available, which is how the picker explains itself. The local rows are always present:

In [ ]:
_orig_avail, _orig_cat = _copilot_available, copilot_catalog
_copilot_available = lambda: True
copilot_catalog = lambda ttl=300: {'test-chat': {'capabilities': {'type': 'chat'}}}
try:
    rows = available_models()
    test_eq([r['value'] for r in rows if r['provider'] == 'litert'], list(LOCAL))
    test_eq([r['value'] for r in rows if r['provider'] == 'copilot'], ['copilot/test-chat'])
finally: _copilot_available, copilot_catalog = _orig_avail, _orig_cat

['gemma-e2b', 'gemma-e4b', 'gemma-12b']

Older coding models are opt-in rather than absent, since `include_legacy=True` is the only way to get a `gpt-4.1` back.

In [ ]:
names = {r['value'] for r in available_models()}
legacy = {r['value'] for r in available_models(include_legacy=True)}
test_eq(names - legacy, set())
sorted(legacy - names)

['openai/gpt-4.1', 'openai/gpt-4.1-mini', 'openai/gpt-4.1-nano']

## Defaults and context windows

LiteRT is the only local runtime. `DEFAULT_POLICY` leaves `turn` unset. That is the user's choice. Points the cheap, frequent jobs at the small local Gemma.

In [ ]:
#| export
DFLT_LOCAL = 'gemma-e4b'
completer = DFLT_LOCAL
cheap = completer          # back-compat alias

#: `None` for the one-shot jobs, which is not "unset" but "whatever `oneshot` says" -- see
#: `Routing.name_for`. Spelling each of them out as the same constant meant the cheap model
#: could only be moved three times or not at all.
DEFAULT_POLICY = {'turn': None, 'oneshot': completer, 'inline': None, 'completion': None,
                  'classify': None, 'summary': None, 'subagent': DFLT_LOCAL}

_LOCAL_CTX = {'gemma-e2b': 16_384, 'gemma-e4b': 16_384, 'gemma-12b': 32_000,
              'qwen-4b': 32_768, 'mini-coder-4b': 32_768, 'ornith-9b': 32_768,
              'llama-qwen-0.6b': 32_768, 'llama-qwen-1.7b': 32_768, 'llama-qwen-4b': 32_768}
DFLT_LOCAL_CTX = 32_768


def local_ctx(name, dflt=DFLT_LOCAL_CTX):
    "The context window for a local model: `$LEELA_LOCAL_CTX` if it says, else the table."
    if (ovr := (env('LOCAL_CTX') or '').strip()):
        if ovr.isdigit(): return int(ovr)
        for part in ovr.split(','):
            k, _, v = part.partition(':')
            if k.strip() == name and v.strip().isdigit(): return int(v)
    return _LOCAL_CTX.get(name, dflt)

Every job in `JOBS` has an entry. A new job cannot be added without deciding where it runs.

In [ ]:
test_eq(set(JOBS) - set(DEFAULT_POLICY), set())
DEFAULT_POLICY

{'turn': None,
 'oneshot': 'gemma-e4b',
 'inline': None,
 'completion': None,
 'classify': None,
 'summary': None,
 'subagent': 'gemma-e4b'}

A local model's context window comes from the table, and `$RAMABANA_LOCAL_CTX` overrides it. Either as one number for everything, or as `name:size` pairs.

In [ ]:
local_ctx('gemma-e4b'), local_ctx('gemma-12b'), local_ctx('a-model-nobody-tabled')

(16384, 32000, 32768)

In [ ]:
os.environ['RAMABANA_LOCAL_CTX'] = 'gemma-e4b:4096'
test_eq(local_ctx('gemma-e4b'), 4096)
test_eq(local_ctx('gemma-12b'), 32_000)          # untouched by a per-model override
del os.environ['RAMABANA_LOCAL_CTX']
local_ctx('gemma-e4b')

16384

## Resolving a model

A `ModelSpec` is a model after every question about it has been answered: which backend runs it, what that backend should be handed, how big its context is, and anything worth telling the user about how that was decided. Everything downstream takes a spec, never a name. The guessing happens once and in one place.

In [ ]:
#| export
@dataclass(frozen=True)
class ModelSpec:
    'One model, resolved: which backend runs it, what to call it, and how big it is.'
    name: str                 # what the user types
    backend: str              # 'rishi' | 'fastllm'
    model_id: str             # what the backend is given
    ctx: int = 128_000        # context window in tokens
    note: str = ''            # anything worth showing about how this was resolved
    config: dict = field(default_factory=dict, compare=False) # runtime options. Never persisted secrets

    @property
    def runtime(self): return self.backend
    @property
    def local(self): return self.backend not in HOSTED
    def __str__(self): return f'{self.name} ({self.model_id})'

def _copilot_ctx(model_id):
    """Context window and a note for a Copilot model. Copilot reports its own. Nothing is guessed.
    Reads the entry here rather than through `rishi.copilot.copilot_ctx`: the catalogue is already
    in hand, and this then needs no rishi newer than the one that fetched it."""
    lim = ((copilot_catalog().get(model_id) or {}).get('capabilities') or {}).get('limits') or {}
    if (n := lim.get('max_prompt_tokens') or lim.get('max_context_window_tokens')): return int(n), ''
    return _cloud_ctx(model_id)      # the same id under its own vendor is the next best answer

def _cloud_ctx(model_id):
    "Context window and a note for a cloud model, from fastllm's tables. Silent about failure."
    try:
        from fastllm.types import get_model_info
        v, _, m = model_id.partition('/')
        info = get_model_info(m or v, v if m else None)
        n = info.get('max_input_tokens') or info.get('max_tokens')
        if n: return int(n), ''
        return 128_000, 'context window unknown, assuming 128k'
    except Exception as e: return 128_000, f'context window unknown ({agent_err(e)}), assuming 128k'

In [ ]:
#| export
#: Prefixes that name a runtime or a transport rather than a vendor, in the spelling that works.
PREFIXES = (*RUNTIMES, 'claude_code')

def prefix_typo(prefix):
    "The known prefix `prefix` is a `-`/`_` slip of, or None."
    key = str(prefix or '').replace('-', '_')
    return next((p for p in PREFIXES if p != prefix and p.replace('-', '_') == key), None)


def resolve(name, default_local=DFLT_LOCAL):
    'A `ModelSpec` for `name`: a short name from the tables, or any full `vendor/model` spec.'
    if not name: name = default_local
    if name in MODELS:
        backend, mid = MODELS[name]
        config = CUSTOM.get(name, {}).get('config', {})
        if backend not in ('remote', 'copilot'):
            if not runtime_available(backend): raise RuntimeError(f'{backend} runtime is unavailable; install rishi[{backend}]')
            ctx = DFLT_AGENT_CTX if backend in AGENTS else local_ctx(name)
            return ModelSpec(name, backend, mid, ctx, config=config)
        if backend == 'copilot':
            ctx, note = _copilot_ctx(mid)
            return ModelSpec(name, backend, mid, ctx, note, config)
        if mid.startswith('claude_code/'):
            ok, error = _load_claude_transport()
            if not ok: raise RuntimeError(f'Claude Code transport unavailable: {error}')
        ctx, note = _cloud_ctx(mid)
        return ModelSpec(name, backend, mid, ctx, note, config)
    if '/' in name:
        runtime, model_id = name.split('/', 1)
        if (fix := prefix_typo(runtime)): raise KeyError(
            f'{name!r}: nothing is spelled {runtime!r} -- did you mean {fix}/{model_id}? An unknown '
            'prefix is read as a vendor and handed to `remote`, which then asks for an API key you '
            'do not need, so the slip surfaces as a credentials error three layers from the typo.')
        if runtime == 'claude_code':
            ok, error = _load_claude_transport()
            if not ok: raise RuntimeError(f'Claude Code transport unavailable: {error}')
        if runtime == 'copilot':
            if not runtime_available('copilot'): raise RuntimeError(COPILOT_UNAVAILABLE)
            ctx, note = _copilot_ctx(model_id)
            return ModelSpec(name, 'copilot', model_id, ctx, note)
        if runtime in ('litert', 'mlx', 'llama', *AGENTS):
            if not runtime_available(runtime): raise RuntimeError(f'{runtime} runtime is unavailable; install rishi[{runtime}]')
            ctx = DFLT_AGENT_CTX if runtime in AGENTS else local_ctx(name)
            return ModelSpec(name, runtime, model_id, ctx)
        ctx, note = _cloud_ctx(name)
        return ModelSpec(name, 'remote', name, ctx, note)
    raise KeyError(f'unknown model {name!r}; known: {", ".join(sorted(MODELS))}, or a vendor/model spec')

@functools.lru_cache(maxsize=256)
def _caps(model_id, runtime):
    "`rishi.model_caps`, memoised. `None` where rishi predates it."
    try:
        from rishi.core import model_caps
        return model_caps(model_id, runtime=runtime)
    except Exception: return None

def spec_caps(spec):
    "What `spec`'s model accepts and what it hands back, or `None` where rishi cannot say."
    return _caps(spec.model_id, spec.backend if spec.local else 'remote')

def accepts(spec, kind):
    "Can `spec`'s model be sent `kind`? Unknown counts as yes."
    c = spec_caps(spec)
    return True if c is None or not c.known else c.accepts(kind)

def model_note(spec):
    "One line about a resolved model, for a status bar."
    where = 'local' if spec.local else 'cloud'
    out = f'{spec.name} · {where} · {spec.ctx//1000}k ctx'
    if (c := spec_caps(spec)) is not None and (m := c.fmt()): out += f' · {m}'
    return out + (f' · {spec.note}' if spec.note else '')

A short name resolves from the tables, and a full `vendor/model` spec resolves even though no table mentions it. A model released this morning is usable this morning.

In [ ]:
resolve('gemma-e4b')

ModelSpec(name='gemma-e4b', backend='litert', model_id='litert-community/gemma-4-E4B-it-litert-lm', ctx=16384, note='', config={})

In [ ]:
spec = resolve('claude_code/claude-sonnet-5')
test_eq(spec.backend, 'remote')
test_eq(spec.local, False)
model_note(spec)

'claude_code/claude-sonnet-5 · cloud · 1000k ctx'

Anything else is a typo, and fails at the point of the typo rather than in the middle of a turn.

In [ ]:
test_fail(lambda: resolve('gpt-9'), contains='unknown model')

Copilot is a runtime of its own, and has to be. Handed to `remote`, `copilot/gpt-5.5` loses its prefix. Rishi reads it as a runtime it already knows and drops it. The bare id goes to whichever vendor owns that name, on that vendor's key. The turn succeeds and the bill is a surprise. The prefix resolves here, and the catalogue is asked for rather than tabled: it is per-plan, it moves, and each entry carries the window Copilot will actually allow.

In [ ]:
# nothing below reaches GitHub: the sign-in and the catalogue are both answered from here
_cat = {'gpt-5.5': {'capabilities': {'type': 'chat', 'limits': {'max_prompt_tokens': 272000}}},
        'claude-opus-4.7': {'capabilities': {'type': 'chat', 'limits': {'max_context_window_tokens': 264000}}},
        'text-embedding-3-small': {'capabilities': {'type': 'embeddings'}}}
_orig_avail, _orig_cat = _copilot_available, copilot_catalog
_copilot_available, copilot_catalog = (lambda: True), (lambda ttl=300: _cat)
try:
    test_eq(runtime_available('copilot'), True)
    test_eq(auth_status()['copilot']['available'], True)

    spec = resolve('copilot/gpt-5.5')
    test_eq(spec.runtime, 'copilot')          # its own runtime, not `remote`
    test_eq(spec.model_id, 'gpt-5.5')         # the prefix named the runtime. It is not part of the id
    test_eq(spec.local, False)                # hosted. It is not sized like something on this disk
    test_eq(spec.ctx, 272000)                 # Copilot's own number
    test_eq(resolve('copilot/claude-opus-4.7').ctx, 264000)   # and its other spelling of one

    # a model Copilot did not name still resolves, on the vendor table, rather than failing here
    assert resolve('copilot/gpt-6-unreleased').ctx > 0

    rows = [r for r in available_models() if r['provider'] == 'copilot']
    test_eq({r['value'] for r in rows}, {'copilot/gpt-5.5', 'copilot/claude-opus-4.7'})
    assert all(r['label'] and r['source'] for r in rows)      # an embedding model cannot answer a turn

finally: _copilot_available, copilot_catalog = _orig_avail, _orig_cat

# and with no sign-in, the ask fails where the typo is rather than three layers downstream
_copilot_available = lambda: False
try: test_fail(lambda: resolve('copilot/gpt-5.5'), contains='rishi[copilot]')
finally: _copilot_available = _orig_avail

## What a model can afford

A briefing is not free, and on a small window it is most of the window. The tool schemas come to 4.7k tokens on a full host, one inlined skill body to 3k more, and the briefing itself to
1.4k. Against the 12.3k a 16k model has before compaction fires. That left room for a single
tool result. The model that ships as the local default could not finish a turn that searched, read, edited and checked: it ran out of window, and `Compactor` had nothing old enough to compact.

`Budget` is that arithmetic, decided from the resolved spec here beside the routing table rather than left to whatever the default constants happen to be. It only ever *withholds*, and nothing it declines is unreachable. A skill body it will not inline is still one `read_skill` away.

In [ ]:
#| export
SMALL_CTX = 24_000       # at or below this window, a model is briefed frugally
TOOL_MAX_FLOOR = 1500    # chars. Below this a file view stops being a file view
FRUGAL_DROP = ('memory', 'web')

@dataclass(frozen=True)
class Budget:
    'What a model can afford to be told, and to be sent back.'
    drop: tuple = ()         # capability groups to withhold from `tools_for`
    inline: bool = True      # whether the briefing may inline a skill body
    tool_max: int = 0        # chars one tool result may spend
    note: str = ''           # why, for a status bar

TAGS_SCHEMA_TOKENS = 3300

def budget_for(spec, tool_max, channel='native'):
    """The briefing `spec`'s model can afford, from its context window.

    `tool_max` is the caller's own default, and this only ever lowers it. Raising the clip on a
    large window would change the case that already works, and the case that does not is the
    only reason this exists.

    A spec with no window, or one we could not read, gets the full briefing. Withholding tools
    from a model whose size is unknown turns not knowing into a smaller agent, and `_cloud_ctx`
    already assumes 128k when a table fails it.
    """
    ctx = getattr(spec, 'ctx', 0) or 0
    if ctx > 0 and channel == 'tags': ctx = max(1, ctx - TAGS_SCHEMA_TOKENS)
    if ctx <= 0 or ctx > SMALL_CTX: return Budget(tool_max=tool_max, note='full briefing')
    mx = min(tool_max, max(TOOL_MAX_FLOOR, (ctx//16)*4))
    return Budget(FRUGAL_DROP, False, mx,
                  f'{ctx//1000}k window: no inlined skills, no {"/".join(FRUGAL_DROP)} tools, '
                  f'tool results clipped to {mx} chars')

In [ ]:
spec = ModelSpec('gemma-e2b', 'litert', 'litert-community/x', 16_384)
b = budget_for(spec, 6000)
test_eq((b.drop, b.inline, b.tool_max), (('memory', 'web'), False, 4096))
b.note

'16k window: no inlined skills, no memory/web tools, tool results clipped to 4096 chars'

In [ ]:
# A 32k local model is briefed in full, and its clip is left where it was.
test_eq(budget_for(ModelSpec('qwen-4b', 'llama', 'x', 32_768), 6000), Budget(tool_max=6000, note='full briefing'))
# An unknown window is not treated as a small one.
test_eq(budget_for(ModelSpec('mystery', 'remote', 'x/y', 0), 6000).inline, True)
# The floor holds for a window small enough that a sixteenth is not a usable view.
test_eq(budget_for(ModelSpec('tiny', 'litert', 'x', 4096), 6000).tool_max, TOOL_MAX_FLOOR)

## Configurable models

`register_model` adds an alias for this process only. Persisting it is the host application's business, since Ramabana does not own the user's configuration file. Passing a Hugging Face URL is enough. Rishi decides which runtime can serve it.

In [ ]:
#| export
def register_model(name, model_id, runtime=None, ctx=128_000, note='custom model', **config):
    "Register a configurable model alias for this process. Persistence belongs to the host app."
    name, model_id = (name or '').strip(), (model_id or '').strip()
    if model_id.startswith(('https://huggingface.co/', 'http://huggingface.co/')):
        model_id = model_id.split('huggingface.co/', 1)[1].strip('/').split('/tree/', 1)[0]
    if not name: name = model_id.rsplit('/', 1)[-1]
    if not name or not model_id: raise ValueError('model name and model id are required')
    if runtime is None:
        from rishi.core import resolve_runtime
        runtime, model_id = resolve_runtime(model_id)
    # `RUNTIMES`, not a literal: `cursor` has to be here or `register_model('grok', 'grok-4.5')`
    # fails after `resolve_runtime` correctly infers it. Rejecting the answer it just asked for.
    if runtime not in RUNTIMES: raise ValueError(f'unknown runtime {runtime!r}')
    if runtime != 'remote' and not runtime_available(runtime):
        raise RuntimeError(f'{runtime} runtime is unavailable; install rishi[{runtime}]')
    MODELS[name] = (runtime, model_id); _LOCAL_CTX[name] = int(ctx or 128_000)
    CUSTOM[name] = {'name': name, 'model_id': model_id, 'runtime': runtime,
                    'ctx': int(ctx or 128_000), 'note': note, 'config': config}
    return resolve(name)

def unregister_model(name):
    "Remove one process-local configurable model alias."
    CUSTOM.pop(name, None); MODELS.pop(name, None); _LOCAL_CTX.pop(name, None)

In [ ]:
register_model('tiny', LOCAL['gemma-e2b'], runtime='litert', ctx=8192)

ModelSpec(name='tiny', backend='litert', model_id='litert-community/gemma-4-E2B-it-litert-lm', ctx=8192, note='', config={})

The alias behaves like a table entry from then on, and `unregister_model` takes it back out of every table it was added to.

In [ ]:
test_eq(resolve('tiny').model_id, LOCAL['gemma-e2b'])
unregister_model('tiny')
test_fail(lambda: resolve('tiny'), contains='unknown model')
'tiny' in MODELS, 'tiny' in CUSTOM

(False, False)

A runtime no engine implements is refused, rather than accepted and discovered later by a turn that cannot start.

In [ ]:
test_fail(lambda: register_model('x', 'some/model', runtime='pytorch'), contains='unknown runtime')

In [ ]:
# a Copilot model can be registered under a short name, which `unknown runtime` used to refuse
_cat = {'gpt-5.5': {'capabilities': {'type': 'chat', 'limits': {'max_prompt_tokens': 272000}}}}
_orig_avail, _orig_cat = _copilot_available, copilot_catalog
_copilot_available, copilot_catalog = (lambda: True), (lambda ttl=300: _cat)
try:
    register_model('cop-test', 'gpt-5.5', runtime='copilot')
    test_eq(resolve('cop-test').runtime, 'copilot')
    test_eq(resolve('cop-test').ctx, 272000)      # sized from the catalogue, not as something local
    test_eq(resolve('cop-test').local, False)
finally:
    _copilot_available, copilot_catalog = _orig_avail, _orig_cat
    MODELS.pop('cop-test', None); CUSTOM.pop('cop-test', None); _LOCAL_CTX.pop('cop-test', None)

## Where the tool schemas travel

A transport's own tool field is the better channel wherever it is open: the schemas are validated, the calls come back structured, and nothing rests on the model minding its punctuation. It is not always open. Claude Code declares tools as an in-process MCP server, and an enterprise-managed configuration forbids every dynamic MCP server there is, which turns a policy about MCP into a coding agent that cannot read a file.

Rishi's `tool_mode='tags'` renders the schemas into the system prompt instead and reads the calls back out of the reply text. That channel cannot be closed. There is no policy against a system prompt. `tool_channel` is the one place that decides, and a Claude Code model becomes a conversational backend like any other rather than one that works only where MCP does.

The agent harnesses sit on both sides of that. Rishi carries their tools natively where their SDK can - an in-process MCP server for Claude Code, custom tools for Cursor - and on tags where it cannot, which is either CLI and any machine whose policy refuses the server. Only the chat knows which of those it ended up on. `tool_channel` asks it whenever there is one and predicts from the spec only when there is not, which is what `budget_for` has to work from: it sizes the tool list, and the tool list is what builds the chat.

It is also the channel every backend already shares: `parse_tool_tags` predates all of this, because Hermes-style tag calls are how the local engines have always called tools. The same briefing is portable across litert, llama, MLX, cursor and hosted models, which is what makes comparing them on one task an experiment with one variable.

In [ ]:
#| export
TOOL_CHANNELS = ('native', 'tags')

_forced_tags = {}   # model_id -> why its wire tool channel is closed on this machine


def force_tags(model_id, why=''):
    "Record that this model's tools cannot travel on the wire here. Later turns stop trying."
    why = why or 'the wire tool channel was refused'
    _forced_tags[str(model_id)] = why
    return why


def forget_forced_tags():
    "Forget what was learned about wire channels. A fixed configuration is tried again."
    _forced_tags.clear()


def tool_channel(spec, chat=None):
    """Which channel a model's tool schemas travel on: `'native'` on the wire, `'tags'` in the
    system prompt. Takes a `ModelSpec` or a bare model id, and the live chat when there is one.

    A chat is the authority. An agent runtime decides its own channel from the path it took and from
    what the harness accepted. No property of the spec can be sure. A Claude chat that opened an
    MCP server and had it refused is on tags now, and only it knows. Without one this predicts, which
    is all `budget_for` can have: it sizes the tool list, and the tool list is what builds the chat.
    """
    if (v := (env('TOOL_CHANNEL') or '').strip().lower()) in TOOL_CHANNELS: return v
    if (ch := getattr(chat, 'tool_channel', None)) in TOOL_CHANNELS: return ch
    mid = str(getattr(spec, 'model_id', spec) or '')
    if (rt := getattr(spec, 'runtime', '')) in AGENTS: return 'native' if _agent_native(rt, spec) else 'tags'
    if mid in _forced_tags: return 'tags'
    return 'tags' if claude_tags(mid) else 'native'

In [ ]:
os.environ.pop('RAMABANA_TOOL_CHANNEL', None)      # stated, not inherited: that is the cell's point
test_eq(tool_channel(ModelSpec('sonnet', 'remote', 'claude-sonnet-4-5', 200_000)), 'native')
os.environ['RAMABANA_TOOL_CHANNEL'] = 'tags'      # any model, for a machine we cannot detect
test_eq(tool_channel('gpt-5.1'), 'tags')
del os.environ['RAMABANA_TOOL_CHANNEL']
force_tags('claude_code/claude-sonnet-5', 'MCP refused by policy')
test_eq(tool_channel('claude_code/claude-sonnet-5'), 'tags')
test_eq(tool_channel('gpt-5.1'), 'native')        # remembered per model, not globally
forget_forced_tags()
# A live chat outranks any of it: an agent runtime picks its channel from the path it took and from
# what the harness accepted, and the spec it was built from cannot know either.
class _Chat:
    def __init__(self, ch): self.tool_channel = ch
test_eq(tool_channel(ModelSpec('c', 'claude', 'claude-opus-5', 200_000), _Chat('native')), 'native')
test_eq(tool_channel(ModelSpec('c', 'claude', 'claude-opus-5', 200_000), _Chat('tags')), 'tags')
# With nothing forced the answer depends on this machine's managed Claude Code config, which is
# not what this cell is about. It states the case it is testing.
_real, _managed_claude_mcp = _managed_claude_mcp, lambda: False
try: test_eq(tool_channel('claude_code/claude-sonnet-5'), 'native')
finally: _managed_claude_mcp = _real

# Which mode a Cursor spec runs in, and the prediction that follows. Cursor runs a tool only in
# agent mode. A chat with tools asks for it. One without has nothing to run and stays read-only.
test_eq((cursor_mode({}), cursor_mode({}, tools=False), cursor_mode({'mode': 'plan'})), ('agent', 'ask', 'plan'))
# Claude is never native: its one channel for a tool it did not ship with is an MCP server, and a
# managed config refuses even the in-process kind. Rishi exposing `tool_channel` does not change that.
import rishi.claude, rishi.cursor
assert getattr(rishi.claude.ClaudeChat, 'tool_channel', None) is not None   # the attribute is not the claim
test_eq(tool_channel(ModelSpec('sonnet', 'claude', 'claude-sonnet-5', 200_000)), 'tags')
# Cursor's depends on the mode it will run in - and on the SDK being reachable at all
_sdk = bool(rishi.cursor.sdk_available())
test_eq(tool_channel(ModelSpec('grok', 'cursor', 'grok-4.5', 128_000)), 'native' if _sdk else 'tags')
test_eq(tool_channel(ModelSpec('grok', 'cursor', 'grok-4.5', 128_000, config={'mode': 'ask'})), 'tags')
# Copilot is not a harness at all: it is chat completions. The schemas go on the wire
test_eq(tool_channel(ModelSpec('cop', 'copilot', 'gpt-5.5', 272_000)), 'native')

## Routing

`Routing` is job to model, and the only thing that decides what runs where. `turn` is the model the user chose. Every other job falls back to it when the policy has nothing to say. Adding a job to `JOBS` cannot strand a caller without a model.

In [ ]:
#| export
@dataclass
class Routing:
    """Job -> model. The policy, and the one place that decides what runs where.
    Environment overrides (`LEELA_MODEL`, `LEELA_MODEL_SUMMARY`, ...) exist.
    """
    turn: str = None
    policy: dict = field(default_factory=lambda: dict(DEFAULT_POLICY))
    default_local: str = DFLT_LOCAL

    def __post_init__(self):
        if not self.turn: self.turn = env('MODEL') or self.default_local
        for job in JOBS:
            if (v := env(f'MODEL_{job.upper()}')): self.policy[job] = v
        self._cache, self.notes = {}, {}

    def name_for(self, job='turn'):
        """The model name `job` runs on: its own policy, then `oneshot` if it is a cheap job, then `turn`."""
        if job == 'turn': return self.turn
        if (n := self.policy.get(job)): return n
        if job in ONESHOT_JOBS and (n := self.policy.get('oneshot')): return n
        return self.turn

    def _resolve(self, name):
        "One resolution, cached: reading fastllm's tables for a cloud model is not free."
        if name not in self._cache: self._cache[name] = resolve(name, self.default_local)
        return self._cache[name]

    def alternatives(self, job):
        "Where `job` goes when its own model is not on this machine, best first."
        seen, out = {self.name_for(job)}, []
        for alt in (self.policy.get('oneshot') if job in ONESHOT_JOBS else None,
                    self.turn, *(self.policy.get(j) for j in JOBS), self.default_local):
            if alt and alt not in seen:
                seen.add(alt); out.append(alt)
        return out

    def spec(self, job='turn', fallback=True):
        """The resolved `ModelSpec` for `job`, on another model when its own is not installed here."""
        n = self.name_for(job)
        try: return self._resolve(n)
        except Exception as e:
            if not fallback or job == 'turn': raise
            for alt in self.alternatives(job):
                try: spec = self._resolve(alt)
                except Exception: continue
                self.notes[job] = f'{n} unavailable ({agent_err(e)}); using {alt}'
                return spec
            raise

    def set(self, name, job='turn'):
        "Point `job` at `name`, validating it first so a typo fails here rather than mid-turn."
        spec = resolve(name, self.default_local)
        if job == 'turn': self.turn = name
        else: self.policy[job] = name
        self._cache[name] = spec
        return spec

    def backends(self):
        "The distinct backend/model pairs this policy needs. An engine is built once and shared."
        out = set()
        for j in JOBS:
            try: s = self.spec(j)
            except Exception: continue      # a job with nowhere to run needs no engine built
            out.add((s.backend, s.model_id))
        return out

    def summary(self):
        "The whole policy in one block, for `/model` with no argument, including anything that moved."
        rows = []
        for j in JOBS:
            try: row = model_note(self.spec(j))
            except Exception as e: row = f'unavailable ({agent_err(e)})'
            rows.append(f'{j:11} {row}' + (f'  [{self.notes[j]}]' if j in self.notes else ''))
        return '\n'.join(rows)

By default the expensive job is the user's model and the cheap ones are local.

In [ ]:
r = Routing(turn='sonnet')
r.name_for('turn'), r.name_for('summary'), r.name_for('completion')

('sonnet', 'gemma-e4b', 'gemma-e4b')

`set` validates before it stores. A misspelled model fails at the `/model` command instead of during the next turn.

In [ ]:
r.set('opus', 'summary')
test_fail(lambda: r.set('sonnnet'), contains='unknown model')
r.name_for('summary')

'opus'

`backends` collapses the policy to the distinct engines it needs, which is what lets one loaded model serve several jobs.

In [ ]:
Routing(turn='sonnet').backends()

{('litert', 'litert-community/gemma-4-E4B-it-litert-lm'),
 ('remote', 'claude_code/claude-sonnet-5')}

`summary` renders the whole policy, for `/model` with no argument.

In [ ]:
print(Routing(turn='sonnet').summary())

turn        sonnet · cloud · 1000k ctx
oneshot     gemma-e4b · local · 16k ctx
inline      gemma-e4b · local · 16k ctx
completion  gemma-e4b · local · 16k ctx
classify    gemma-e4b · local · 16k ctx
summary     gemma-e4b · local · 16k ctx
subagent    gemma-e4b · local · 16k ctx


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()